# Results Analysis and Standardization

This notebook standardizes benchmark assets, verifies GPU usage signals, and generates comparison charts across OpenMP, MPI, and CUDA.

## 1. Set Up Paths, Folders, and Configuration

Define paths, expected folder structure, and architecture labels. Also create any missing directories and list empty ones.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os

ROOT = Path.cwd()
DATA_INPUT = ROOT / "data" / "input" / "pgm"
RAW_INPUT = ROOT / "data" / "input" / "raw"
BENCHMARKS = ROOT / "benchmarks" / "raw" / "benchmarks.csv"
PLOTS_OUT = ROOT / "plots" / "output"
RESULTS_OUT = ROOT / "results" / "tables"

ARCH_LABELS = {
    "sequential": "Sequential",
    "openmp": "OpenMP (CPU)",
    "openmp_target": "OpenMP Target (GPU)",
    "mpi": "MPI (CPU)",
    "mpi_cuda": "MPI + CUDA (GPU)",
    "cuda": "CUDA",
}

expected_dirs = [
    DATA_INPUT,
    RAW_INPUT,
    ROOT / "benchmarks" / "raw",
    ROOT / "benchmarks" / "processed",
    ROOT / "plots" / "output",
    ROOT / "results" / "images",
    ROOT / "results" / "tables",
]

for d in expected_dirs:
    d.mkdir(parents=True, exist_ok=True)

empty_dirs = [d for d in expected_dirs if not any(d.iterdir())]

print("Root:", ROOT)
print("Empty directories:")
for d in empty_dirs:
    print(" -", d)

## 2. Download and Verify Benchmark Images

This section downloads the configured images (if missing) and captures basic metadata.

In [ ]:
from pathlib import Path

pgm_files = sorted(DATA_INPUT.glob("*.pgm"))

if not pgm_files:
    print("No PGM files found. Run: uv run python scripts/fetch_images.py")
else:
    def read_pgm_shape(path: Path):
        with path.open("rb") as f:
            if f.readline().strip() != b"P5":
                return None
            def next_token():
                token = f.readline()
                while token.startswith(b"#"):
                    token = f.readline()
                return token
            dims = next_token().split()
            while len(dims) < 2:
                dims += next_token().split()
            width, height = map(int, dims)
            return width, height

    for p in pgm_files:
        shape = read_pgm_shape(p)
        size_kb = p.stat().st_size / 1024
        print(f"{p.name:20s}  shape={shape}  size={size_kb:.1f} KB")

## 3. Inspect Dataset and Standardize Sizes

Summarize the size distribution and bucket sizes for fair comparisons.

In [ ]:
import pandas as pd

records = []
for p in pgm_files:
    shape = read_pgm_shape(p)
    if not shape:
        continue
    width, height = shape
    records.append({"image": p.stem, "width": width, "height": height, "pixels": width * height})

sizes_df = pd.DataFrame(records).sort_values("pixels")
if sizes_df.empty:
    print("No size metadata available.")
else:
    display(sizes_df)
    print("Size buckets:")
    display(sizes_df.groupby("pixels").size().rename("count").reset_index())

## 4. Load Benchmark Reports and Normalize Metrics

Load `benchmarks.csv` and normalize architecture names and units.

In [ ]:
import numpy as np

if not BENCHMARKS.exists():
    print("Benchmarks not found. Run: ./scripts/benchmark_all")
    df = pd.DataFrame()
else:
    df = pd.read_csv(BENCHMARKS)
    df["arch"] = df["implementation"].map(ARCH_LABELS).fillna(df["implementation"])
    df["ms"] = df["seconds"] * 1000
    display(df.head())

## 5. Detect GPU Usage and Runtime Environment

Infer GPU usage from benchmark entries and note whether CUDA data is present.

In [ ]:
if df.empty:
    print("No benchmark data loaded.")
else:
    has_cuda = (df["implementation"] == "cuda").any()
    print("CUDA rows present:", has_cuda)
    if has_cuda:
        display(df[df["implementation"] == "cuda"].head())
    else:
        print("No CUDA data found. Verify build/cuda_convolution and rerun benchmarks.")

## 6. Aggregate Results by Architecture (OpenMP, MPI, CUDA)

Compute best runtime and speedup per image and architecture.

In [ ]:
if df.empty:
    best = pd.DataFrame()
else:
    best = (
        df.groupby(["image", "implementation"], as_index=False)
          .agg(seconds=("seconds", "min"))
    )
    seq = best[best["implementation"] == "sequential"].set_index("image")["seconds"]
    best["speedup"] = best.apply(lambda r: seq.get(r["image"], 1.0) / r["seconds"], axis=1)
    display(best.sort_values(["image", "implementation"]))

## 7. Generate Comparison Charts

Create charts for scaling and cross-architecture comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

if not df.empty:
    omp_df = df[df["implementation"] == "openmp"]
    mpi_df = df[df["implementation"] == "mpi"]

    if not omp_df.empty:
        plt.figure(figsize=(8, 5))
        sns.lineplot(data=omp_df, x="threads", y="seconds", hue="image", marker="o")
        plt.title("OpenMP Scaling (per image)")
        plt.tight_layout()
        plt.show()

    if not mpi_df.empty:
        plt.figure(figsize=(8, 5))
        sns.lineplot(data=mpi_df, x="ranks", y="seconds", hue="image", marker="o")
        plt.title("MPI Scaling (per image)")
        plt.tight_layout()
        plt.show()

    if not best.empty:
        plt.figure(figsize=(9, 5))
        sns.barplot(data=best, x="image", y="seconds", hue="implementation")
        plt.title("Best Runtime by Architecture (per image)")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(9, 5))
        sns.barplot(data=best, x="image", y="speedup", hue="implementation")
        plt.title("Best Speedup by Architecture (per image)")
        plt.tight_layout()
        plt.show()

### CPU vs GPU Comparison

OpenMP and MPI are CPU-based in this project. Here we compare the best CPU result against CUDA.

In [ ]:
if not df.empty:
    cpu_best = (
        best[best["implementation"].isin(["sequential", "openmp", "mpi"])]
        .groupby("image", as_index=False)
        .agg(seconds=("seconds", "min"))
        .assign(implementation="cpu_best")
    )
    cuda_best = best[best["implementation"] == "cuda"].copy()

    if not cuda_best.empty:
        cpu_gpu = pd.concat([cpu_best, cuda_best], ignore_index=True)
        plt.figure(figsize=(9, 5))
        sns.barplot(data=cpu_gpu, x="image", y="seconds", hue="implementation")
        plt.title("CPU Best vs CUDA Runtime")
        plt.tight_layout()
        plt.show()

        merged = cpu_best.merge(cuda_best, on="image", suffixes=("_cpu", "_cuda"))
        merged["cpu_over_cuda"] = merged["seconds_cpu"] / merged["seconds_cuda"]
        plt.figure(figsize=(9, 5))
        sns.barplot(data=merged, x="image", y="cpu_over_cuda")
        plt.title("CPU Best / CUDA Speedup Ratio")
        plt.tight_layout()
        plt.show()
    else:
        print("CUDA data not found in benchmarks.")

## 8. Validate Empty Folders and Cleanup Report

List empty folders and export a summary CSV for the report if needed.

In [ ]:
if best.empty:
    print("No aggregated benchmark results to export.")
else:
    RESULTS_OUT.mkdir(parents=True, exist_ok=True)
    out_path = RESULTS_OUT / "best_by_architecture.csv"
    best.sort_values(["image", "implementation"]).to_csv(out_path, index=False)
    print("Wrote", out_path)

print("Empty directories detected:")
for d in empty_dirs:
    print(" -", d)